In [4]:
import numpy as np
import pandas as pd

from pathlib import Path

In [5]:
datasource_path = Path("..")/"src/02_homework/car_fuel_efficiency_2026.csv"

In [7]:
df = pd.read_csv(datasource_path)

print(df.shape)
df.head()

(10000, 11)


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


# Question 1
There's one column with missing values. What is it?

* `'engine_displacement'`
* `'horsepower'`
* `'vehicle_weight'`
* `'model_year'`

In [39]:
list_features = ["engine_displacement", "horsepower", "vehicle_weight", "model_year"]
df[list_features].isnull().sum(axis=0)

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
dtype: int64

In [26]:
y_col = "fuel_efficiency_mpg"

In [28]:
df[y_col].describe()

count    10000.000000
mean        29.997700
std          2.930245
min         19.800000
25%         28.000000
50%         30.000000
75%         31.900000
max         41.200000
Name: fuel_efficiency_mpg, dtype: float64

# Question 2

What's the median (50% percentile) for variable `'horsepower'`?

- 204
- 254
- 304
- 354


In [15]:
df["horsepower"].median(skipna=True)

np.float64(254.0)

# Question 3

* We need to deal with missing values for the column from Q1.
* We have two options: fill it with 0 or with the mean of this variable.
* Try both options. For each, train a linear regression model without regularization using the code from the lessons.
* For computing the mean, use the training only!
* Use the validation dataset to evaluate the models and compare the RMSE of each option.
* Round the RMSE scores to 3 decimal digits using `round(score, 3)`. This
  keeps the imputation difference visible in this release.
* Which option gives better RMSE?

Options:

- With 0
- With mean
- Both are equally good

In [79]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

In [80]:
null_col = "horsepower"

In [57]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

def rmse(y, y_pred):
    se = (y - y_pred) ** 2
    mse = se.mean()
    return np.sqrt(mse)

In [83]:
df_val[f"{null_col}_na_zero"] = df_val[null_col].fillna(0)
df_val[f"{null_col}_na_mean"] = df_val[null_col].fillna(df_train[null_col].mean())

df_val

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg,horsepower_na_zero,horsepower_na_mean
2480,2004,USA,Gasoline,Rear-wheel drive,3,2240,6,265.0,3990,18.5,30.4,265.0,265.0
289,2019,Asia,Gasoline,All-wheel drive,5,2390,6,259.0,4770,18.7,32.8,259.0,259.0
6086,2018,USA,Gasoline,Rear-wheel drive,2,2360,6,265.0,4320,19.9,27.8,265.0,265.0
3075,2008,USA,Gasoline,Rear-wheel drive,3,2430,6,279.0,4430,17.9,28.6,279.0,279.0
8123,1999,USA,Gasoline,Front-wheel drive,2,2220,5,249.0,4030,21.0,31.1,249.0,249.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1638,2020,Europe,Gasoline,Rear-wheel drive,3,1980,5,229.0,4140,16.7,30.3,229.0,229.0
5891,1989,USA,Gasoline,All-wheel drive,3,2590,7,263.0,4630,18.8,28.7,263.0,263.0
7427,2017,USA,Hybrid,Front-wheel drive,3,2390,7,264.0,4550,19.0,30.6,264.0,264.0
608,2014,USA,Gasoline,Front-wheel drive,2,2330,6,244.0,4440,19.9,31.0,244.0,244.0


In [46]:
# df_val[["engine_displacement", "horsepower_na_zero", "vehicle_weight", "model_year"]]
# df_val[["engine_displacement", "horsepower_na_mean", "vehicle_weight", "model_year"]]

In [64]:

list_features_a = ["engine_displacement", "horsepower_na_zero", "vehicle_weight", "model_year"]

w0_a, w_a = train_linear_regression(
    X=df_val[list_features_a],
    y=df_val[y_col]
)

print(w0_a, w_a)

y_pred_a = w0_a + df_val[list_features_a].dot(w_a)

round(rmse(
    y=df_val[y_col],
    y_pred=y_pred_a
), 3)

-155.9810314669499 [-0.00087544 -0.0016548  -0.00400707  0.10280457]


np.float64(2.2)

In [65]:
list_features_b = ["engine_displacement", "horsepower_na_mean", "vehicle_weight", "model_year"]

w0_b, w_b = train_linear_regression(
    X=df_val[list_features_b],
    y=df_val[y_col]
)

print(w0_b, w_b)

y_pred_b = w0_b + df_val[list_features_b].dot(w_b)

round(rmse(
    y=df_val[y_col],
    y_pred=y_pred_b
), 3)

-150.13357422341497 [-0.00037039 -0.00837806 -0.0039363   0.10002879]


np.float64(2.199)

# Question 4

* Now let's train a regularized linear regression.
* For this question, fill the NAs with 0. 
* Try different values of `r` from this list: `[0, 0.01, 0.1, 1, 5, 10, 100]`.
* Use RMSE to evaluate the model on the validation dataset.
* Round the RMSE scores to 4 decimal digits. This keeps the small but real
  regularization differences visible instead of turning several choices into a tie.
* Which `r` gives the best RMSE?

If multiple options give the same best RMSE, select the smallest `r`.

Options:

- 0
- 0.01
- 0.1
- 1
- 5
- 10
- 100



In [81]:
def train_linear_regression_reg(X, y, r=0.001):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])

    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

In [85]:
list_r = [0, 0.01, 0.1, 1, 5, 10, 100]
list_features_a = ["engine_displacement", "horsepower_na_zero", "vehicle_weight", "model_year"]

for n, r_iter in enumerate(list_r, 1):
    print(n, f"r={r_iter}")

    w0_a, w_a = train_linear_regression_reg(
        X=df_val[list_features_a],
        y=df_val[y_col],
        r=r_iter
    )

    y_pred_a = w0_a + df_val[list_features_a].dot(w_a)

    rmse_val = round(
        rmse(
            y=df_val[y_col],
            y_pred=y_pred_a
        ), 4)

    print(f"rmse= {rmse_val}\n")
    print("#"*30)

1 r=0
rmse= 2.1997

##############################
2 r=0.01
rmse= 2.2023

##############################
3 r=0.1
rmse= 2.2674

##############################
4 r=1
rmse= 2.3923

##############################
5 r=5
rmse= 2.4179

##############################
6 r=10
rmse= 2.4214

##############################
7 r=100
rmse= 2.4247

##############################



# Question 5 

* We used seed 42 for splitting the data. Let's find out how selecting the seed influences our score.
* Try different seed values: `[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]`.
* For each seed, do the train/validation/test split with 60%/20%/20% distribution.
* Fill the missing values with 0 and train a model without regularization.
* For each seed, evaluate the model on the validation dataset and collect the RMSE scores. 
* What's the standard deviation of all the scores? To compute the standard deviation, use `np.std`.
* Round the result to 3 decimal digits (`round(std, 3)`)

What's the value of std?

- 0.006
- 0.016
- 0.029
- 0.036

> Note: Standard deviation shows how different the values are.
> If it's low, then all values are approximately the same.
> If it's high, the values are different. 
> If standard deviation of scores is low, then our model is *stable*.




# Question 6 (For Q6, use seed `9`.)

* Split the dataset like previously, use seed 9.
* Combine train and validation datasets.
* Fill the missing values with 0 and train a model with `r=0.001`. 
* What's the RMSE on the test dataset?

In [66]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(9)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

In [67]:
def train_linear_regression_reg(X, y, r=0.001):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])

    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

In [ ]:
df_train_q6 = pd.concat([df_train, df_val], axis=0)
df_train_q6[f"{null_col}_na_zero"] = df_train_q6[null_col].fillna(0)

df_train_q6

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg,horsepower_na_zero
3644,2014,USA,Gasoline,Front-wheel drive,3,2320,6,246.0,4100,19.0,32.5,246.0
9184,1998,Europe,Diesel,Rear-wheel drive,4,2230,6,252.0,4250,NaN,33.2,252.0
520,2008,Europe,Gasoline,Front-wheel drive,3,2700,6,276.0,4490,17.8,32.4,276.0
5685,2004,USA,Gasoline,Front-wheel drive,2,2220,7,269.0,4060,20.0,29.9,269.0
2401,1979,USA,Gasoline,All-wheel drive,3,2370,6,NaN,4690,22.0,26.4,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
8981,2023,USA,Hybrid,Front-wheel drive,4,2690,7,324.0,4510,17.9,35.7,324.0
3978,1998,Asia,Gasoline,Front-wheel drive,5,1920,5,190.0,4000,19.0,30.8,190.0
6198,1994,USA,Gasoline,Rear-wheel drive,3,1800,5,225.0,3980,19.6,29.1,225.0
1102,2009,Europe,Gasoline,Front-wheel drive,2,1970,5,232.0,3560,15.3,32.2,232.0


In [75]:
df_test[f"{null_col}_na_zero"] = df_test[null_col].fillna(0)

df_test

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg,horsepower_na_zero
5463,2011,USA,Gasoline,Rear-wheel drive,4,2100,6,258.0,4060,16.7,29.0,258.0
9878,2011,Asia,Hybrid,Front-wheel drive,3,2160,6,223.0,4130,19.7,35.7,223.0
8503,1975,Europe,Gasoline,Front-wheel drive,4,2190,6,NaN,4030,17.4,26.2,0.0
2571,1997,Europe,Diesel,Front-wheel drive,4,2320,6,257.0,4640,19.8,31.6,257.0
6851,1988,Asia,Hybrid,Rear-wheel drive,5,2330,6,235.0,4130,20.7,30.4,235.0
...,...,...,...,...,...,...,...,...,...,...,...,...
6200,1996,Europe,Gasoline,Front-wheel drive,4,2260,6,250.0,4170,21.8,33.3,250.0
501,2015,Asia,Gasoline,Rear-wheel drive,4,2300,6,249.0,4360,17.4,29.9,249.0
6782,1982,USA,Diesel,Front-wheel drive,3,2240,6,265.0,4170,17.4,31.0,265.0
4444,1978,USA,Diesel,Rear-wheel drive,3,2380,6,278.0,4210,17.1,27.2,278.0


In [76]:

list_features_a = ["engine_displacement", "horsepower_na_zero", "vehicle_weight", "model_year"]
r = 1e-3

w0_a, w_a = train_linear_regression_reg(
    X=df_train_q6[list_features_a],
    y=df_train_q6[y_col],
    r=r
)

print(w0_a, w_a)

y_pred_a = w0_a + df_test[list_features_a].dot(w_a)

round(rmse(
    y=df_test[y_col],
    y_pred=y_pred_a
), 3)

-152.37535772493302 [-0.00135534 -0.00042645 -0.00401765  0.10140402]


np.float64(2.236)